# ScrubRef on ImageWoof

## 1. Environment and Experiment Setup
This section initializes the execution environment, verifies runtime dependencies,
prepares persistent storage on Google Drive, downloads or links the dataset, and loads the experiment configuration.

### 1.1 Google Colab Setup

The following cell mounts Google Drive and prepares the repository for
execution in Colab. Skip this cell when running the notebook locally.


In [ ]:
from google.colab import drive
from pathlib import Path
import os
import subprocess
import sys


os.chdir("/content")
drive.mount("/content/drive")

GITHUB_REPO = "https://github.com/imnima2005/scrubref-unlearning.git"
REPO_DIR_NAME = "scrubref-unlearning"

DRIVE_ROOT = Path("/content/drive/MyDrive")
REPO_DIR = DRIVE_ROOT / REPO_DIR_NAME

if not REPO_DIR.exists():
    print("Cloning repository...")
    result = subprocess.run(
        ["git", "clone", GITHUB_REPO, str(REPO_DIR)],
        capture_output=True,
        text=True
    )
    if result.returncode != 0:
        print("Error:")
        print(result.stderr)
        raise RuntimeError("Clone failed")
    print("Clone successful!")
else:
    print(f"already exist: {REPO_DIR}")

os.chdir(REPO_DIR)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Project root:", Path.cwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
already exist: /content/drive/MyDrive/scrubref-unlearning
Project root: /content/drive/MyDrive/scrubref-unlearning


### 1.2 Runtime Diagnostics

Reports the Python version, platform, and installed package versions without
modifying or installing any packages. Also checks for available GPU devices.


In [35]:
# Colab environment check — do not install or downgrade packages here.
import importlib.metadata as metadata
import platform
import sys

PACKAGES = (
    "numpy",
    "pandas",
    "scipy",
    "scikit-learn",
    "matplotlib",
    "tensorflow",
    "joblib",
)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())

for package in PACKAGES:
    try:
        version = metadata.version(package)
    except metadata.PackageNotFoundError:
        version = "not installed"

    print(f"{package}: {version}")

print("Environment check complete.")

Python: 3.13.16
Platform: Linux-6.6.159+-x86_64-with-glibc2.39
numpy: 2.4.6
pandas: 2.2.3
scipy: 1.16.3
scikit-learn: 1.6.1
matplotlib: 3.10.0
tensorflow: 2.21.0
joblib: 1.6.0
Environment check complete.


In [36]:
from __future__ import annotations

import ctypes
import gc
import json
import math
import os
import random
import shutil
import tarfile
import time
import urllib.request
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple, Union

try:
    import joblib
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import tensorflow as tf
except ValueError as exc:
    if "numpy.dtype size changed" in str(exc):
        raise RuntimeError(
            "The Colab VM contains an incompatible NumPy/Pandas binary stack. "
            "Use Runtime > Disconnect and delete runtime, reconnect, and run this notebook "
            "without executing any pip-install cell."
        ) from exc
    raise
from sklearn import set_config as sklearn_set_config
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    log_loss,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

sklearn_set_config(working_memory=256)

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))


TensorFlow: 2.21.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


### 1.3 Persistent Keras Cache

Creates a symbolic link to store Keras datasets permanently in Google Drive,
preventing re-downloading across sessions.


In [37]:
import shutil
from pathlib import Path
import os

# Configure persistent Keras datasets directory directly in your project's data folder
DRIVE_DATASETS = REPO_DIR / "data"
LOCAL_DATASETS = Path.home() / ".keras" / "datasets"

DRIVE_DATASETS.mkdir(parents=True, exist_ok=True)
LOCAL_DATASETS.parent.mkdir(parents=True, exist_ok=True)

# Replace only the ephemeral local cache directory with a Drive-backed symlink
if LOCAL_DATASETS.is_symlink():
    if LOCAL_DATASETS.resolve() != DRIVE_DATASETS.resolve():
        LOCAL_DATASETS.unlink()
elif LOCAL_DATASETS.exists():
    shutil.rmtree(LOCAL_DATASETS)

if not LOCAL_DATASETS.exists():
    os.symlink(str(DRIVE_DATASETS), str(LOCAL_DATASETS), target_is_directory=True)

os.environ["KERAS_HOME"] = str(REPO_DIR)

print("Persistent Keras cache:", DRIVE_DATASETS)
print("Local cache link:", LOCAL_DATASETS, "->", LOCAL_DATASETS.resolve())

Persistent Keras cache: /content/drive/MyDrive/scrubref-unlearning/data
Local cache link: /root/.keras/datasets -> /content/drive/MyDrive/scrubref-unlearning/data


### 1.4 Dataset Preparation

Checks, downloads from verified mirrors, verifies via MD5 checksum, and extracts
the dataset into the persistent Google Drive folder. Skips download if the dataset
is already complete.


In [38]:
# Prepare CIFAR-100 from a fast mirror and keep it permanently in Google Drive.
from pathlib import Path
import hashlib
import shutil
import subprocess
import tarfile


def file_md5(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.md5()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


# Use the same DRIVE_DATASETS path (REPO_DIR / "data") configured in the symlink cell
main_dataset_dir = REPO_DIR / "data"
dataset_dir = main_dataset_dir / "cifar100"
dataset_dir.mkdir(parents=True, exist_ok=True)

local_archive = Path("/content/cifar-100-python.tar.gz")
drive_archive = dataset_dir / "cifar-100-python.tar.gz"
extracted_dir = dataset_dir / "cifar-100-python"
expected_md5 = "eb9058c3a382ffc7106e4002c42a8d85"

mirror_urls = [
    "https://huggingface.co/datasets/nakroy/cifar100-python/resolve/main/cifar-100-python.tar.gz",
    "https://data.brainchip.com/dataset-mirror/cifar100/cifar-100-python.tar.gz",
]

required_extracted_files = [
    extracted_dir / "train",
    extracted_dir / "test",
    extracted_dir / "meta",
]

if all(path.exists() for path in required_extracted_files):
    print("CIFAR-100 already prepared:", extracted_dir)
else:
    if drive_archive.exists() and file_md5(drive_archive) != expected_md5:
        print("Removing invalid Drive archive:", drive_archive)
        drive_archive.unlink()

    if not drive_archive.exists():
        local_archive.unlink(missing_ok=True)
        downloaded = False
        for url in mirror_urls:
            print("Trying:", url)
            result = subprocess.run(
                [
                    "wget",
                    "-c",
                    "--progress=bar:force:noscroll",
                    "--timeout=30",
                    "--tries=3",
                    "-O",
                    str(local_archive),
                    url,
                ],
                check=False,
            )
            if result.returncode != 0 or not local_archive.exists():
                continue
            actual_md5 = file_md5(local_archive)
            print("MD5:", actual_md5)
            if actual_md5 == expected_md5:
                shutil.copy2(local_archive, drive_archive)
                downloaded = True
                break
            local_archive.unlink(missing_ok=True)

        if not downloaded:
            raise RuntimeError("All CIFAR-100 mirrors failed or returned an invalid archive.")

    print("Extracting CIFAR-100 into Google Drive...")
    with tarfile.open(drive_archive, "r:gz") as archive:
        try:
            archive.extractall(dataset_dir, filter="data")
        except TypeError:
            archive.extractall(dataset_dir)

    missing = [str(path) for path in required_extracted_files if not path.exists()]
    if missing:
        raise RuntimeError("CIFAR-100 extraction is incomplete:\n" + "\n".join(missing))
    print("Dataset prepared:", extracted_dir)

local_archive.unlink(missing_ok=True)

Trying: https://huggingface.co/datasets/nakroy/cifar100-python/resolve/main/cifar-100-python.tar.gz
MD5: eb9058c3a382ffc7106e4002c42a8d85
Extracting CIFAR-100 into Google Drive...
Dataset prepared: /content/drive/MyDrive/scrubref-unlearning/data/cifar100/cifar-100-python


### 1.5 Experiment Configuration

Loads the YAML experiment configuration and applies settings such as the run mode
(e.g., debug vs. full execution).


In [66]:
#from src.config import ExperimentConfig

#CFG = ExperimentConfig.from_yaml(REPO_DIR / "configs" / "cifar10.yaml")

#CFG.run_mode = "debug"
#CFG.apply_mode()
from src.config import ExperimentConfig
from src import config as config_module
from src import data_loader as data_loader_module

CFG = ExperimentConfig.from_yaml(REPO_DIR / "configs" / "cifar10.yaml")
CFG.run_mode = "debug"
CFG.apply_mode()

# Make CFG available inside the source modules
config_module.CFG = CFG
data_loader_module.CFG = CFG

# Make other notebook-level variables available if needed
data_loader_module.IN_COLAB = True
data_loader_module.REPO_DIR = REPO_DIR

print("CFG injected successfully")
print("run_mode:", CFG.run_mode)
print("dataset:", CFG.dataset)

CFG injected successfully
run_mode: debug
dataset: cifar10


### 1.6 Reproducibility and Runtime Initialization

This step initializes the experiment's reproducibility settings before any data processing or model training begins. The random seed is read from the active experiment configuration and applied globally to reduce run-to-run variation.

Reusable utilities for setting determinism and managing TensorFlow memory are implemented in `src/utils.py`. The notebook only imports and invokes the required utility.


In [40]:
from src.utils import set_global_determinism

set_global_determinism(CFG.seeds[0])

## 2. Dataset Preparation and Unlearning Split

This section loads the configured dataset, creates a reproducible retain/forget split, and prepares the TensorFlow input pipelines used by training and evaluation.

Dataset-specific loading, split construction, image decoding, augmentation, and `tf.data` pipeline logic are implemented in `src/data_loader.py`.

### 2.1 Dataset Loading and Validation

Load the dataset specified by the experiment configuration and validate that the expected files and class structure are available. The loader returns a `DataBundle` containing the training and test examples, labels, class names, and image metadata.


In [41]:
from src.data_loader import (
    DataBundle,
    ensure_imagewoof,
    load_dataset,
    list_directory_split,
    load_directory_bundle
)

### 2.2 Reproducible Retain/Forget Split

Partition the training and test data into retain and forget subsets according to the configured forget classes. A validation subset is sampled from the retain-training data using the experiment seed, so the split can be reproduced across runs.


In [42]:
from src.data_loader import (
    UnlearningSplit,
    make_unlearning_split,
)

### 2.3 Input Pipeline and Augmentation

Build TensorFlow datasets for training and evaluation. Images are decoded or resized as needed, optional training augmentation is applied, and batching, shuffling, determinism, and prefetching are configured through the experiment settings.


In [43]:
from src.data_loader import (
    decode_item,
    augment_image,
    make_dataset,
    materialize_items,
)

## 3. Model and Unlearning Method

This section defines the model architecture and the training utilities used to prepare the original model and support subsequent unlearning experiments. The reusable implementation is organized in `src/models.py` and `src/training.py`, while the notebook documents the experimental setup and orchestrates model loading or training.


### 3.1 Model Architecture and Model Utilities

Define the model architecture and reusable model-inspection utilities. These functions support model construction, checkpoint cloning, output interpretation, feature extraction, and selective control of trainable layers.


In [49]:
from src.models import (
    build_resnet18,
    classifier_layer,
    clone_architecture,
    clone_with_weights,
    fresh_clone,
    leaf_layers,
    model_containers,
    output_to_logits,
    output_to_probs,
    set_all_trainable,
    set_tail_trainable,
)

### 3.2 Training and Original Model Initialization

Load a compatible pretrained checkpoint when available; otherwise, initialize and train the original model using the configured dataset and training parameters. The original model serves as the reference point for the subsequent unlearning procedure.


In [50]:
from src.training import (
    compile_supervised,
    infer_output_is_logits,
    train_supervised,
    load_or_train_original,
)

### 3.3 Synthetic Proxy Data Generation

Generate class-conditioned synthetic proxy images by optimizing input images against the frozen teacher model. The target classes are assigned in a balanced schedule, and the resulting images, labels, and generation metadata are cached for reuse.

The proxy-generation procedure is implemented in `src/proxy_generation.py`. The notebook records the configuration and invokes the generator.


In [51]:
from src.proxy_generation import (
    balanced_proxy_targets,
    generate_class_proxies,
    generate_unsir_error_maximizing_noise
)

### 3.4 ScrubRef Unlearning Procedure

Apply the ScrubRef unlearning procedure to the original model using the retain split and class-conditioned proxy samples. The student is initialized from the original model and optimized to preserve utility on retained data while updating its predictions—and optionally its representations—on proxy samples associated with the forget classes.

The implementation is provided in `src/scrubref.py`. The notebook records the configuration and invokes the procedure; evaluation is reported separately in Section 4.


In [52]:
from src.scrubref import (
    scrubref_train
)

### 3.5 Baseline Unlearning Methods

Implement comparison methods used to assess the ScrubRef procedure. Each baseline starts from the same original model and uses the same unlearning split and evaluation protocol unless its method-specific definition requires otherwise. Baseline implementations are organized under `src/baselines/`; the notebook configures and runs the selected methods.


In [53]:
from src.baselines.simple import (
    logit_clamp_baseline,
    head_reset_baseline,
    retain_finetune_baseline,
    retrain_baseline,
)

from src.baselines.distillation import (
    bad_teacher_baseline,
    scrub_baseline
)

from src.baselines.ssd import (
    ssd_baseline,
    estimate_fisher_diagonal
)

from src.baselines.unsir import (
    unsir_baseline
)


## 4. Evaluation and Unlearning Assessment

Evaluate the utility, calibration, privacy-related behavior, and parameter changes of the trained or unlearned model. Reusable evaluation and metric implementations are defined in `src/metrics.py`; the notebook calls them and presents the resulting metrics.

Utility metrics measure predictive performance on retained and forgotten classes. Membership inference metrics assess whether forget-set training examples can be distinguished from forget-set test examples. Parameter-change statistics quantify how much the model changed during unlearning.


### 4.1 Prediction and Utility Metrics

Generate class probabilities and compute predictive performance metrics, including overall, retained, and forgotten accuracy, top-5 accuracy, macro-F1, balanced accuracy, negative log-likelihood, and per-class accuracy.


### 4.2 Calibration Metrics

Measure the quality of the model's predicted probabilities using Expected Calibration Error (ECE) and the multiclass Brier score. These complement accuracy by evaluating confidence calibration.


### 4.3 Membership Inference Evaluation

Evaluate membership inference on forgotten examples by comparing forget-set training members with forget-set test non-members. Report AUC-based scores from confidence, loss, entropy, modified entropy, and a learned attack, along with the learned attack's average precision.


### 4.4 Parameter and Runtime Statistics

Quantify the model's parameter changes relative to its original version and, when available, record peak GPU memory usage. These statistics help characterize the unlearning procedure's computational and model-level effects.


In [55]:
def predict_probabilities(
    model: tf.keras.Model,
    items: np.ndarray,
    labels: np.ndarray,
    bundle: DataBundle,
    output_is_logits: bool,
    batch_size: int,
    seed: int,
) -> Tuple[np.ndarray, np.ndarray]:
    ds = make_dataset(items, labels, bundle, batch_size, False, False, False, seed)
    probabilities: List[np.ndarray] = []
    labels_out: List[np.ndarray] = []
    for xb, yb in ds:
        output = model(xb, training=False)
        probabilities.append(output_to_probs(output, output_is_logits).numpy())
        labels_out.append(yb.numpy())
    return np.concatenate(probabilities), np.concatenate(labels_out)


def expected_calibration_error(y_true: np.ndarray, probs: np.ndarray, bins: int = 15) -> float:
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    correctness = (predictions == y_true).astype(np.float64)
    boundaries = np.linspace(0.0, 1.0, bins + 1)
    ece = 0.0
    for lower, upper in zip(boundaries[:-1], boundaries[1:]):
        mask = (confidences > lower) & (confidences <= upper)
        if np.any(mask):
            ece += np.abs(correctness[mask].mean() - confidences[mask].mean()) * mask.mean()
    return float(ece)


def brier_multiclass(y_true: np.ndarray, probs: np.ndarray, num_classes: int) -> float:
    one_hot = np.eye(num_classes, dtype=np.float64)[y_true]
    return float(np.mean(np.sum((probs - one_hot) ** 2, axis=1)))


def per_class_accuracy(y_true: np.ndarray, y_pred: np.ndarray, num_classes: int) -> np.ndarray:
    result = np.full(num_classes, np.nan, dtype=np.float64)
    for class_id in range(num_classes):
        mask = y_true == class_id
        if np.any(mask):
            result[class_id] = np.mean(y_pred[mask] == class_id)
    return result


def evaluate_utility(
    model: tf.keras.Model,
    bundle: DataBundle,
    forget_classes: Sequence[int],
    output_is_logits: bool,
    batch_size: int,
    seed: int,
) -> Tuple[Dict[str, float], np.ndarray]:
    probs, y_true = predict_probabilities(
        model, bundle.test_items, bundle.test_labels, bundle, output_is_logits, batch_size, seed
    )
    y_pred = probs.argmax(axis=1)
    forget_mask = np.isin(y_true, np.asarray(forget_classes))
    retain_mask = ~forget_mask
    top5 = np.mean([
        label in np.argpartition(row, -min(5, bundle.num_classes))[-min(5, bundle.num_classes):]
        for row, label in zip(probs, y_true)
    ])
    metrics = {
        "overall_accuracy": float(accuracy_score(y_true, y_pred)),
        "retained_accuracy": float(accuracy_score(y_true[retain_mask], y_pred[retain_mask])),
        "forgotten_accuracy": float(accuracy_score(y_true[forget_mask], y_pred[forget_mask])),
        "top5_accuracy": float(top5),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "nll": float(log_loss(y_true, np.clip(probs, 1e-7, 1.0), labels=np.arange(bundle.num_classes))),
        "brier": brier_multiclass(y_true, probs, bundle.num_classes),
        "ece_15": expected_calibration_error(y_true, probs, bins=15),
        "mean_global_forget_mass": float(np.mean(np.sum(probs[:, forget_classes], axis=1))),
        "mean_forget_test_true_class_confidence": float(
            np.mean(probs[np.where(forget_mask)[0], y_true[forget_mask]])
        ),
    }
    return metrics, per_class_accuracy(y_true, y_pred, bundle.num_classes)


def entropy_scores(probs: np.ndarray) -> np.ndarray:
    probs = np.clip(probs, 1e-7, 1.0 - 1e-7)
    return -np.sum(probs * np.log(probs), axis=1)


def modified_entropy_scores(probs: np.ndarray, labels: np.ndarray) -> np.ndarray:
    probs = np.clip(probs, 1e-7, 1.0 - 1e-7)
    rows = np.arange(len(labels))
    true_probs = probs[rows, labels]
    first = -(1.0 - true_probs) * np.log(true_probs)
    mask = np.ones_like(probs, dtype=bool)
    mask[rows, labels] = False
    remaining = -np.sum((probs * np.log(1.0 - probs)) * mask, axis=1)
    return first + remaining


def safe_auc(y_true: np.ndarray, scores: np.ndarray) -> float:
    if len(np.unique(y_true)) < 2:
        return float("nan")
    auc = roc_auc_score(y_true, scores)
    return float(max(auc, 1.0 - auc))


def learned_mia(
    member_probs: np.ndarray,
    member_labels: np.ndarray,
    nonmember_probs: np.ndarray,
    nonmember_labels: np.ndarray,
    repeats: int,
    seed: int,
) -> Tuple[float, float]:
    def attack_features(probs: np.ndarray, labels: np.ndarray) -> np.ndarray:
        rows = np.arange(len(labels))
        true_conf = probs[rows, labels]
        entropy = entropy_scores(probs)
        mod_entropy = modified_entropy_scores(probs, labels)
        sorted_probs = np.sort(probs, axis=1)[:, -3:]
        return np.column_stack([true_conf, entropy, mod_entropy, sorted_probs])

    x = np.concatenate([
        attack_features(member_probs, member_labels),
        attack_features(nonmember_probs, nonmember_labels),
    ])
    y = np.concatenate([
        np.ones(len(member_probs), dtype=np.int64),
        np.zeros(len(nonmember_probs), dtype=np.int64),
    ])
    aucs: List[float] = []
    aps: List[float] = []
    for repeat in range(repeats):
        x_train, x_test, y_train, y_test = train_test_split(
            x, y, test_size=0.5, random_state=seed + repeat, stratify=y
        )
        attack = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced"))
        attack.fit(x_train, y_train)
        scores = attack.predict_proba(x_test)[:, 1]
        aucs.append(roc_auc_score(y_test, scores))
        aps.append(average_precision_score(y_test, scores))
    return float(np.mean(aucs)), float(np.mean(aps))


def evaluate_membership_inference(
    model: tf.keras.Model,
    split: UnlearningSplit,
    bundle: DataBundle,
    output_is_logits: bool,
    batch_size: int,
    repeats: int,
    seed: int,
) -> Dict[str, float]:
    member_probs, member_labels = predict_probabilities(
        model,
        split.forget_train_items,
        split.forget_train_labels,
        bundle,
        output_is_logits,
        batch_size,
        seed,
    )
    nonmember_probs, nonmember_labels = predict_probabilities(
        model,
        split.forget_test_items,
        split.forget_test_labels,
        bundle,
        output_is_logits,
        batch_size,
        seed,
    )
    n = min(len(member_probs), len(nonmember_probs))
    rng = np.random.default_rng(seed)
    member_idx = rng.choice(len(member_probs), n, replace=False)
    nonmember_idx = rng.choice(len(nonmember_probs), n, replace=False)
    member_probs = member_probs[member_idx]
    member_labels = member_labels[member_idx]
    nonmember_probs = nonmember_probs[nonmember_idx]
    nonmember_labels = nonmember_labels[nonmember_idx]

    probs = np.concatenate([member_probs, nonmember_probs])
    labels = np.concatenate([member_labels, nonmember_labels])
    membership = np.concatenate([np.ones(n), np.zeros(n)])
    rows = np.arange(len(labels))
    true_conf = probs[rows, labels]
    negative_loss = np.log(np.clip(true_conf, 1e-7, 1.0))
    negative_entropy = -entropy_scores(probs)
    negative_modified_entropy = -modified_entropy_scores(probs, labels)
    learned_auc, learned_ap = learned_mia(
        member_probs, member_labels, nonmember_probs, nonmember_labels, repeats, seed
    )
    return {
        "mia_confidence_auc": safe_auc(membership, true_conf),
        "mia_loss_auc": safe_auc(membership, negative_loss),
        "mia_entropy_auc": safe_auc(membership, negative_entropy),
        "mia_modified_entropy_auc": safe_auc(membership, negative_modified_entropy),
        "mia_learned_auc": learned_auc,
        "mia_learned_ap": learned_ap,
        "mia_member_count": float(n),
    }


def parameter_change_statistics(original: tf.keras.Model, changed: tf.keras.Model, tolerance: float = 1e-7) -> Dict[str, float]:
    original_weights = original.get_weights()
    changed_weights = changed.get_weights()
    if len(original_weights) != len(changed_weights):
        return {"changed_parameter_fraction": float("nan"), "relative_l2_parameter_change": float("nan")}
    total = 0
    changed_count = 0
    squared_delta = 0.0
    squared_original = 0.0
    for before, after in zip(original_weights, changed_weights):
        delta = after.astype(np.float64) - before.astype(np.float64)
        total += delta.size
        changed_count += int(np.count_nonzero(np.abs(delta) > tolerance))
        squared_delta += float(np.sum(delta ** 2))
        squared_original += float(np.sum(before.astype(np.float64) ** 2))
    return {
        "changed_parameter_fraction": float(changed_count / max(total, 1)),
        "relative_l2_parameter_change": float(math.sqrt(squared_delta) / max(math.sqrt(squared_original), 1e-12)),
    }


def peak_gpu_memory_mb() -> float:
    try:
        info = tf.config.experimental.get_memory_info("GPU:0")
        return float(info["peak"] / (1024 ** 2))
    except Exception:
        return float("nan")

## 5. Experiment Pipeline Components

This section defines the reusable components used by the experiment pipeline, including experiment configuration, method orchestration, evaluation, and artifact generation. The notebook invokes the completed pipeline in Section 8; implementation details are organized in `src/experiment_runner.py`.



### 5.1 Seed-Level Experiment Runner

Define the workflow for one dataset and random seed, including split construction, original-model loading, proxy generation, method execution, evaluation, and artifact storage. The runner is invoked later in Section 8.


In [56]:
from src.experiment_runner import (
    build_method,
    method_model_path,
    load_saved_model,
    run_seed,
    save_json,
)

## 6. Representation Audit and Analysis

Import representation auditing utilities from `src.representation_audit` to compute linear CKA, Jensen-Shannon divergence on output probability distributions, and linear probe accuracy on retained vs. forgotten classes.


### 6.1 Representation Audit

Extract and cache train, test, and CKA features for the configured methods. Compare each method with the original model and, when available, the retrained reference using linear CKA, representation probes, predictive-distribution divergence, and prediction disagreement. The audit implementation is kept in the experiment-analysis modules; the notebook launches it and inspects the resulting metrics.


In [57]:
from src.representation_audit import (
    stratified_indices,
    extract_features,
    linear_cka,
    representation_probe_metrics,
    run_representation_audit
)

### 6.2 Per-Seed Figures

Visualize the per-seed results for retained and forgotten accuracy, membership-inference AUC, calibration, and representation similarity. Figures are saved in the run's `figures/` directory and provide a visual summary of the corresponding metric CSV files.


In [58]:
from src.representation_audit import (
    make_seed_figures,
)

## 7. Runtime and Dataset Preflight

Verify that the dataset can be loaded with the expected image data types and estimate its memory footprint before starting the experiments. Release the temporary dataset bundle after the check.


In [61]:
from src.utils import array_memory_mb, release_memory

PREFLIGHT_BUNDLE = load_dataset(CFG)
print("Train dtype:", PREFLIGHT_BUNDLE.train_items.dtype)
print("Test dtype:", PREFLIGHT_BUNDLE.test_items.dtype)
print(
    "Dataset image RAM:",
    f"{array_memory_mb(PREFLIGHT_BUNDLE.train_items, PREFLIGHT_BUNDLE.test_items):.2f} MB",
)
if not PREFLIGHT_BUNDLE.path_mode and PREFLIGHT_BUNDLE.train_items.dtype != np.uint8:
    raise RuntimeError("RAM preflight failed: CIFAR images are not uint8.")
del PREFLIGHT_BUNDLE
release_memory(clear_session=False)
print("RAM preflight passed.")


Train dtype: uint8
Test dtype: uint8
Dataset image RAM: 175.78 MB
RAM preflight passed.


In [ ]:
import inspect
print(inspect.getsource(load_or_train_original))

In [ ]:
CFG.enabled_methods = [
    "original", "retrained", "logit_clamp",
    "head_reset", "retain_finetune"
    # "scrubref_head"  <-- Temporarily comment out or delete this section.
]

In [ ]:
import os
import pandas as pd
csv_path = Path(CFG.results_root) / CFG.dataset / f"seed_{42}" / "utility_metrics.csv"

#csv_path = "/content/drive/MyDrive/scrubref_revision_results/cifar100/seed_42/utility_metrics.csv"

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    cols = ["method", "overall_accuracy", "retained_accuracy", "forgotten_accuracy", "macro_f1", "nll"]
    available_cols = [c for c in cols if c in df.columns]
    display(df[available_cols])
else:
    print(f"⚠️ Result file not found at: {csv_path}")
    print("Please execute the experiment runner cell first (under 'Execute experiments') to generate the results.")

## 8. Execute Experiments

Run the experiment pipeline for each configured seed. Existing results may be reused when `FORCE_RERUN` is `False`; set it to `True` to force the pipeline to rerun instead of relying on cached results.


In [67]:
FORCE_RERUN = False

SEED_OUTPUTS = []
for experiment_seed in CFG.seeds:
    print(f"\n{'=' * 90}\nRunning {CFG.dataset}, seed={experiment_seed}\n{'=' * 90}")
    SEED_OUTPUTS.append(run_seed(CFG, experiment_seed, force_rerun=FORCE_RERUN))

print("\n" + "="*90 + "\nAll experiments completed successfully!\n" + "="*90)
SEED_OUTPUTS


Running cifar10, seed=42
Dataset storage: uint8 175.78 MB
RAM-safe split prepared: retain_train=40500, retain_val=4500, forget_train=5000, retain_test=9000, forget_test=1000
Bundle + split image arrays: 351.56 MB
Epoch 1/3
782/782 - 91s - 116ms/step - accuracy: 0.4914 - loss: 1.4051
Epoch 2/3
782/782 - 77s - 98ms/step - accuracy: 0.6791 - loss: 0.9083
Epoch 3/3
782/782 - 78s - 100ms/step - accuracy: 0.7572 - loss: 0.6986
Completed original {'overall_accuracy': 0.686, 'retained_accuracy': 0.7345555555555555, 'forgotten_accuracy': 0.249, 'top5_accuracy': 0.9633, 'macro_f1': 0.6735049879461779, 'balanced_accuracy': 0.6860000000000002, 'nll': 1.0283967974331418, 'brier': 0.4475856896555336, 'ece_15': 0.11155272021889688, 'mean_global_forget_mass': 0.03085394948720932, 'mean_forget_test_true_class_confidence': 0.2339777946472168}
Epoch 1/3
633/633 - 76s - 120ms/step - accuracy: 0.5016 - loss: 1.3658 - val_accuracy: 0.5927 - val_loss: 1.1300
Epoch 2/3
633/633 - 65s - 103ms/step - accuracy: 

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c


All experiments completed successfully!


[{'run_dir': PosixPath('results/cifar10/seed_42'),
  'utility': PosixPath('results/cifar10/seed_42/utility_metrics.csv'),
  'mia': PosixPath('results/cifar10/seed_42/mia_metrics.csv'),
  'runtime': PosixPath('results/cifar10/seed_42/runtime_metrics.csv'),
  'representation': PosixPath('results/cifar10/seed_42/representation_metrics.csv')}]

## 9. Aggregate all seeds

The aggregation cell produces mean, standard deviation, and 95% normal-approximation confidence intervals for every numeric metric.

In [68]:
def aggregate_metric_file(cfg: ExperimentConfig, filename: str, output_name: str) -> pd.DataFrame:
    frames = []
    for seed in cfg.seeds:
        path = Path(cfg.results_root) / cfg.dataset / f"seed_{seed}" / filename
        if path.exists():
            frames.append(pd.read_csv(path))
    if not frames:
        raise FileNotFoundError(f"No {filename} files were found.")
    combined = pd.concat(frames, ignore_index=True)
    numeric_columns = [column for column in combined.select_dtypes(include=[np.number]).columns if column != "seed"]
    grouped = combined.groupby("method")[numeric_columns].agg(["mean", "std", "count"])
    rows = []
    for method in grouped.index:
        row: Dict[str, object] = {"method": method}
        for metric in numeric_columns:
            mean = grouped.loc[method, (metric, "mean")]
            std = grouped.loc[method, (metric, "std")]
            count = grouped.loc[method, (metric, "count")]
            ci95 = 1.96 * std / math.sqrt(count) if count > 1 and np.isfinite(std) else float("nan")
            row[f"{metric}_mean"] = mean
            row[f"{metric}_std"] = std
            row[f"{metric}_ci95"] = ci95
        rows.append(row)
    output = pd.DataFrame(rows)
    aggregate_dir = Path(cfg.results_root) / cfg.dataset / "aggregate"
    aggregate_dir.mkdir(parents=True, exist_ok=True)
    combined.to_csv(aggregate_dir / f"all_seeds_{filename}", index=False)
    output.to_csv(aggregate_dir / output_name, index=False)
    return output


AGGREGATED_UTILITY = aggregate_metric_file(CFG, "utility_metrics.csv", "utility_summary.csv")
AGGREGATED_MIA = aggregate_metric_file(CFG, "mia_metrics.csv", "mia_summary.csv")
AGGREGATED_RUNTIME = aggregate_metric_file(CFG, "runtime_metrics.csv", "runtime_summary.csv")
AGGREGATED_REPRESENTATION = aggregate_metric_file(CFG, "representation_metrics.csv", "representation_summary.csv")

print("Utility summary")
display(AGGREGATED_UTILITY)
print("MIA summary")
display(AGGREGATED_MIA)
print("Representation summary")
display(AGGREGATED_REPRESENTATION)

Utility summary


,method,overall_accuracy_mean,overall_accuracy_std,overall_accuracy_ci95,retained_accuracy_mean,retained_accuracy_std,retained_accuracy_ci95,forgotten_accuracy_mean,forgotten_accuracy_std,forgotten_accuracy_ci95,...,ece_15_ci95,mean_global_forget_mass_mean,mean_global_forget_mass_std,mean_global_forget_mass_ci95,mean_forget_test_true_class_confidence_mean,mean_forget_test_true_class_confidence_std,mean_forget_test_true_class_confidence_ci95,scrub_final_loss_mean,scrub_final_loss_std,scrub_final_loss_ci95
0,bad_teacher,0.7592,NaN,NaN,0.843556,NaN,NaN,0.000,NaN,NaN,...,NaN,1.257917e-02,NaN,NaN,7.139730e-02,NaN,NaN,NaN,NaN,NaN
1,head_reset,0.7239,NaN,NaN,0.804333,NaN,NaN,0.000,NaN,NaN,...,NaN,1.453367e-05,NaN,NaN,1.296963e-05,NaN,NaN,NaN,NaN,NaN
2,logit_clamp,0.6625,NaN,NaN,0.736111,NaN,NaN,0.000,NaN,NaN,...,NaN,9.473412e-12,NaN,NaN,2.556195e-11,NaN,NaN,NaN,NaN,NaN
3,original,0.6860,NaN,NaN,0.734556,NaN,NaN,0.249,NaN,NaN,...,NaN,3.085395e-02,NaN,NaN,2.339778e-01,NaN,NaN,NaN,NaN,NaN
4,retain_finetune,0.7650,NaN,NaN,0.850000,NaN,NaN,0.000,NaN,NaN,...,NaN,8.226274e-06,NaN,NaN,3.590140e-05,NaN,NaN,NaN,NaN,NaN
5,retrained,0.6800,NaN,NaN,0.755556,NaN,NaN,0.000,NaN,NaN,...,NaN,6.063175e-07,NaN,NaN,6.835522e-07,NaN,NaN,NaN,NaN,NaN
6,scrub,0.5471,NaN,NaN,0.607889,NaN,NaN,0.000,NaN,NaN,...,NaN,9.685133e-05,NaN,NaN,2.287625e-06,NaN,NaN,NaN,NaN,NaN
7,scrubref_head,0.7318,NaN,NaN,0.810556,NaN,NaN,0.023,NaN,NaN,...,NaN,8.495534e-03,NaN,NaN,4.171356e-02,NaN,NaN,0.598458,NaN,NaN
8,scrubref_rep,0.7478,NaN,NaN,0.830889,NaN,NaN,0.000,NaN,NaN,...,NaN,4.385206e-03,NaN,NaN,2.431520e-02,NaN,NaN,0.609229,NaN,NaN
9,ssd,0.1000,NaN,NaN,0.111111,NaN,NaN,0.000,NaN,NaN,...,NaN,4.185120e-02,NaN,NaN,4.187708e-02,NaN,NaN,NaN,NaN,NaN


MIA summary


,method,mia_confidence_auc_mean,mia_confidence_auc_std,mia_confidence_auc_ci95,mia_loss_auc_mean,mia_loss_auc_std,mia_loss_auc_ci95,mia_entropy_auc_mean,mia_entropy_auc_std,mia_entropy_auc_ci95,...,mia_modified_entropy_auc_ci95,mia_learned_auc_mean,mia_learned_auc_std,mia_learned_auc_ci95,mia_learned_ap_mean,mia_learned_ap_std,mia_learned_ap_ci95,mia_member_count_mean,mia_member_count_std,mia_member_count_ci95
0,bad_teacher,0.506099,NaN,NaN,0.506099,NaN,NaN,0.506440,NaN,NaN,...,NaN,0.474839,NaN,NaN,0.480490,NaN,NaN,1000.0,NaN,NaN
1,head_reset,0.510250,NaN,NaN,0.510225,NaN,NaN,0.514785,NaN,NaN,...,NaN,0.478064,NaN,NaN,0.485018,NaN,NaN,1000.0,NaN,NaN
2,logit_clamp,0.505057,NaN,NaN,0.500000,NaN,NaN,0.503105,NaN,NaN,...,NaN,0.500230,NaN,NaN,0.500331,NaN,NaN,1000.0,NaN,NaN
3,original,0.500070,NaN,NaN,0.500070,NaN,NaN,0.501103,NaN,NaN,...,NaN,0.490763,NaN,NaN,0.497198,NaN,NaN,1000.0,NaN,NaN
4,retain_finetune,0.503099,NaN,NaN,0.503110,NaN,NaN,0.519331,NaN,NaN,...,NaN,0.500179,NaN,NaN,0.504577,NaN,NaN,1000.0,NaN,NaN
5,retrained,0.510326,NaN,NaN,0.508763,NaN,NaN,0.527012,NaN,NaN,...,NaN,0.510984,NaN,NaN,0.507128,NaN,NaN,1000.0,NaN,NaN
6,scrub,0.506020,NaN,NaN,0.503495,NaN,NaN,0.505543,NaN,NaN,...,NaN,0.498802,NaN,NaN,0.499971,NaN,NaN,1000.0,NaN,NaN
7,scrubref_head,0.513391,NaN,NaN,0.513391,NaN,NaN,0.516032,NaN,NaN,...,NaN,0.495087,NaN,NaN,0.498873,NaN,NaN,1000.0,NaN,NaN
8,scrubref_rep,0.510232,NaN,NaN,0.510232,NaN,NaN,0.508479,NaN,NaN,...,NaN,0.515126,NaN,NaN,0.508545,NaN,NaN,1000.0,NaN,NaN
9,ssd,0.518628,NaN,NaN,0.518629,NaN,NaN,0.516522,NaN,NaN,...,NaN,0.505185,NaN,NaN,0.500271,NaN,NaN,1000.0,NaN,NaN


Representation summary


,method,cka_to_original_mean,cka_to_original_std,cka_to_original_ci95,linear_probe_overall_accuracy_mean,linear_probe_overall_accuracy_std,linear_probe_overall_accuracy_ci95,linear_probe_retained_accuracy_mean,linear_probe_retained_accuracy_std,linear_probe_retained_accuracy_ci95,...,js_to_retrained_all_ci95,js_to_retrained_retain_mean,js_to_retrained_retain_std,js_to_retrained_retain_ci95,js_to_retrained_forget_mean,js_to_retrained_forget_std,js_to_retrained_forget_ci95,prediction_disagreement_to_retrained_mean,prediction_disagreement_to_retrained_std,prediction_disagreement_to_retrained_ci95
0,bad_teacher,0.849083,NaN,NaN,0.8325,NaN,NaN,0.838889,NaN,NaN,...,NaN,0.086910,NaN,NaN,0.221456,NaN,NaN,0.261333,NaN,NaN
1,logit_clamp,1.000000,NaN,NaN,0.7870,NaN,NaN,0.800000,NaN,NaN,...,NaN,0.121810,NaN,NaN,0.223756,NaN,NaN,0.326667,NaN,NaN
2,original,1.000000,NaN,NaN,0.7870,NaN,NaN,0.800000,NaN,NaN,...,NaN,0.123770,NaN,NaN,0.282008,NaN,NaN,0.330667,NaN,NaN
3,retrained,0.823193,NaN,NaN,0.7725,NaN,NaN,0.793333,NaN,NaN,...,NaN,0.000000,NaN,NaN,0.000000,NaN,NaN,0.000000,NaN,NaN
4,scrub,0.169144,NaN,NaN,0.7590,NaN,NaN,0.758889,NaN,NaN,...,NaN,0.237311,NaN,NaN,0.419291,NaN,NaN,0.457333,NaN,NaN
5,scrubref_head,1.000000,NaN,NaN,0.7870,NaN,NaN,0.800000,NaN,NaN,...,NaN,0.093085,NaN,NaN,0.203732,NaN,NaN,0.276000,NaN,NaN
6,scrubref_rep,0.887548,NaN,NaN,0.8225,NaN,NaN,0.841667,NaN,NaN,...,NaN,0.096695,NaN,NaN,0.196807,NaN,NaN,0.257333,NaN,NaN
7,ssd,0.177614,NaN,NaN,0.4165,NaN,NaN,0.426667,NaN,NaN,...,NaN,0.443069,NaN,NaN,0.276710,NaN,NaN,0.893333,NaN,NaN
8,unsir,0.000000,NaN,NaN,0.2600,NaN,NaN,0.262222,NaN,NaN,...,NaN,0.602731,NaN,NaN,0.683090,NaN,NaN,0.904000,NaN,NaN


## 10. Reviewer-focused result checks

These checks do not decide acceptance. They identify invalid or weak result patterns before values are copied into the manuscript.

In [69]:
def reviewer_readiness_checks(cfg: ExperimentConfig) -> pd.DataFrame:
    aggregate_dir = Path(cfg.results_root) / cfg.dataset / "aggregate"
    utility = pd.read_csv(aggregate_dir / "utility_summary.csv").set_index("method")
    mia = pd.read_csv(aggregate_dir / "mia_summary.csv").set_index("method")
    representation = pd.read_csv(aggregate_dir / "representation_summary.csv").set_index("method")
    rows = []
    for method in ["scrubref_head", "scrubref_rep"]:
        if method not in utility.index:
            continue
        retain_drop = utility.loc["original", "retained_accuracy_mean"] - utility.loc[method, "retained_accuracy_mean"]
        forgotten_accuracy = utility.loc[method, "forgotten_accuracy_mean"]
        learned_mia = mia.loc[method, "mia_learned_auc_mean"]
        cka_original = representation.loc[method, "cka_to_original_mean"]
        cka_retrained = representation.loc[method, "cka_to_retrained_mean"]
        linear_probe_forget = representation.loc[method, "linear_probe_forgotten_accuracy_mean"]
        rows.append({
            "method": method,
            "retained_accuracy_drop": retain_drop,
            "forgotten_accuracy": forgotten_accuracy,
            "learned_mia_auc": learned_mia,
            "cka_to_original": cka_original,
            "cka_to_retrained": cka_retrained,
            "closer_to_retrained_than_original": bool(cka_retrained > cka_original),
            "linear_probe_forgotten_accuracy": linear_probe_forget,
            "output_forgetting_pass": bool(forgotten_accuracy <= 0.05),
            "empirical_mia_pass": bool(learned_mia <= 0.60),
            "utility_pass": bool(retain_drop <= 0.03),
        })
    checks = pd.DataFrame(rows)
    checks.to_csv(aggregate_dir / "reviewer_readiness_checks.csv", index=False)
    return checks


REVIEWER_CHECKS = reviewer_readiness_checks(CFG)
display(REVIEWER_CHECKS)

,method,retained_accuracy_drop,forgotten_accuracy,learned_mia_auc,cka_to_original,cka_to_retrained,closer_to_retrained_than_original,linear_probe_forgotten_accuracy,output_forgetting_pass,empirical_mia_pass,utility_pass
0,scrubref_head,-0.076000,0.023,0.495087,1.000000,0.823193,False,0.67,True,True,True
1,scrubref_rep,-0.096333,0.000,0.515126,0.887548,0.848232,False,0.65,True,True,True


## Optional ablation runner

The main result must be compared with mechanisms that can trivially suppress outputs. This ablation executes the two ScrubRef variants under four settings:

- no logit surgery, no clamp;
- logit surgery only;
- clamp only;
- both mechanisms.

In [ ]:
def run_scrubref_ablations(base_cfg: ExperimentConfig, seed: int) -> pd.DataFrame:
    rows = []
    settings = [
        (False, False, "none"),
        (True, False, "logit_surgery"),
        (False, True, "confidence_clamp"),
        (True, True, "surgery_and_clamp"),
    ]
    for use_surgery, use_clamp, label in settings:
        ablation_cfg = ExperimentConfig(**asdict(base_cfg))
        ablation_cfg.seeds = (seed,)
        ablation_cfg.enabled_methods = ("scrubref_head", "scrubref_rep")
        ablation_cfg.representation_methods = ("original", "retrained", "scrubref_head", "scrubref_rep")
        ablation_cfg.enable_main_method_logit_surgery = use_surgery
        ablation_cfg.enable_main_method_clamp = use_clamp
        ablation_cfg.results_root = str(Path(base_cfg.results_root) / "ablations" / label)
        outputs = run_seed(ablation_cfg, seed, force_rerun=False)
        utility = pd.read_csv(outputs["utility"])
        utility["ablation"] = label
        rows.append(utility)
    result = pd.concat(rows, ignore_index=True)
    output_dir = Path(base_cfg.results_root) / "ablations"
    output_dir.mkdir(parents=True, exist_ok=True)
    result.to_csv(output_dir / f"{base_cfg.dataset}_ablation_results.csv", index=False)
    return result


RUN_ABLATIONS = True
ABLATION_RESULTS = run_scrubref_ablations(CFG, CFG.seeds[0]) if RUN_ABLATIONS else pd.DataFrame()
display(ABLATION_RESULTS)

## Output package

After completion, the aggregate directory contains:

- `utility_summary.csv`
- `mia_summary.csv`
- `runtime_summary.csv`
- `representation_summary.csv`
- `reviewer_readiness_checks.csv`
- all per-seed result files, model checkpoints, proxy arrays, and figures

Use only values generated by these files in the revised manuscript. Do not reuse numerical results from the submitted notebook because its scrub branch and membership evaluation were invalid.

## Method-label discipline

- `scrubref_head` is the corrected original algorithmic scope.
- `scrubref_rep` is an algorithmic extension and must be described as such.
- `unsir`, `bad_teacher`, `scrub`, and `ssd` are TensorFlow-native adaptations. Before publication, compare their settings and outputs against the official repositories and state any implementation differences.
- Membership-inference values are empirical audit results, not formal privacy guarantees.

## RAM-safe execution notes

- Use **Runtime → Run all** after a clean Colab runtime is connected.
- Keep `FORCE_RERUN = False` to resume completed methods.
- Do not convert complete CIFAR arrays to `float32`; conversion occurs per mini-batch.
- The publication configuration uses three seeds and all enabled methods.
- If Colab disconnects, reconnect, run from the top, and completed model checkpoints will be reused.


## Synthetic proxy generation

The original notebook assigned zero class weight to every synthetic sample, so the scrub branch contributed zero loss. Here, proxy samples are optimized through the frozen original model and the notebook asserts that the forget branch produces a non-zero gradient.

## Representation audits

The audit measures whether forgotten classes remain recoverable from the penultimate representation. CKA is computed in feature space, avoiding the quadratic sample-by-sample Gram matrix.

## Baselines

The notebook labels TensorFlow-native adaptations as such. For final publication, report the exact configuration used and verify the adaptation against each official implementation.

## Corrected ScrubRef objectives

The head-only and representation-aware variants use the same output-level objective. The representation-aware variant additionally changes a small tail of the backbone, preserves retained features, and moves proxy features toward the retained feature centroid.

## Experiment runner

Each method is saved and evaluated immediately so Colab does not keep every model in GPU or CPU memory. The runner clears TensorFlow and Python allocations between methods and seeds. Existing model files are loaded unless `force_rerun=True`, so interrupted publication runs can resume.
